In [1]:
from pathlib import Path

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

FIGSIZE = (20,18)
DPI = 100
GENERATE_PLOTS = False

Repo root: /Users/jedrek/Documents/Studium Volkswirschaftslehre/4. Semester/DEDA Project/DEDA_LLM_Spatial_Hotelling


In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
import sys

sys.path.insert(0, str(REPO_ROOT / 'src'))

from hotelling.spatial.census import build_grid_polygons
from hotelling.spatial.assembly import (
    build_demand_grid,
    enrich_supermarkets_with_brw,
)

PATH_RAW       = REPO_ROOT / 'data' / 'raw'
PATH_PROCESSED = REPO_ROOT / 'data' / 'processed'

# Load grid (convert point midpoints to square polygons)
grid = gpd.read_parquet(PATH_PROCESSED / 'pop_grid.parquet')
grid = build_grid_polygons(grid)
grid['index'] = grid.index
print(f"Grid: {len(grid)} cells")

# Load all spatial layer inputs
grid_malls         = gpd.read_parquet(PATH_PROCESSED / 'grid_malls.parquet')
grid_with_stations = gpd.read_parquet(PATH_PROCESSED / 'grid_with_stations.parquet')
travel_times       = pd.read_parquet(PATH_PROCESSED / 'travel_times.parquet')
employment_clusters= gpd.read_parquet(PATH_PROCESSED / 'employment_clusters.parquet')
supermarkets       = gpd.read_parquet(PATH_PROCESSED / 'supermarkets.parquet')
brw                = gpd.read_file(PATH_RAW / 'brw_2025.gpkg')
print("All inputs loaded.")

Grid: 16170 cells
All inputs loaded.


In [3]:
# Assemble demand grid: flags + employment + travel times + MSS/ESIx + normalization
demand_grid = build_demand_grid(
    grid=grid,
    grid_malls=grid_malls,
    grid_with_stations=grid_with_stations,
    travel_times=travel_times,
    employment_clusters=employment_clusters,
    mss_path=PATH_RAW / 'mss.gpkg',
    esix_path=PATH_RAW / 'esix.gpkg',
    output_path=PATH_PROCESSED / 'demand_grid.parquet',
)

# Enrich incumbents with BRW data
supermarkets_full = enrich_supermarkets_with_brw(
    supermarkets=supermarkets,
    brw=brw,
    output_path=PATH_PROCESSED / 'supermarkets_full.parquet',
)

print(f"demand_grid: {len(demand_grid)} cells, {len(demand_grid.columns)} columns")
print(f"supermarkets_full: {len(supermarkets_full)} stores")

/Users/jedrek/Documents/Studium Volkswirschaftslehre/4. Semester/DEDA Project/DEDA_LLM_Spatial_Hotelling/src/hotelling/spatial/assembly.py:572: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: df.set_index("to_id")["travel_time"].to_dict())


demand_grid: 32348 cells, 39 columns
supermarkets_full: 533 stores


In [5]:
demand_grid

,x_mp_100m,y_mp_100m,geometry,GITTER_ID_100m,Einwohner,index,has_mall,has_station,station_class,matched_db_station,...,kom,plr_id_esix,plr,plr_name_right,esix_wert,esix_rang,esix_schicht,esix_normalized,si_normalized,station_class_normalized
0,4543150,3266250,"POLYGON ((4543200 3266200, 4543200 3266300, 45...",CRS3035RES100mN3266250E4543150,0,0,False,False,NaN,None,...,gültig,06040808,06040808,Hüttenweg,1.3510,16.0,1,0.974803,0.829389,NaN
1,4543250,3266250,"POLYGON ((4543300 3266200, 4543300 3266300, 45...",CRS3035RES100mN3266250E4543250,0,1,False,False,NaN,None,...,gültig,06040808,06040808,Hüttenweg,1.3510,16.0,1,0.974803,0.829389,NaN
2,4543350,3266250,"POLYGON ((4543400 3266200, 4543400 3266300, 45...",CRS3035RES100mN3266250E4543350,0,2,False,False,NaN,None,...,gültig,06040808,06040808,Hüttenweg,1.3510,16.0,1,0.974803,0.829389,NaN
3,4543450,3266250,"POLYGON ((4543500 3266200, 4543500 3266300, 45...",CRS3035RES100mN3266250E4543450,0,3,False,False,NaN,None,...,gültig,06040810,06040810,Dahlem,1.2539,24.0,1,0.955231,0.829389,NaN
3,4543450,3266250,"POLYGON ((4543500 3266200, 4543500 3266300, 45...",CRS3035RES100mN3266250E4543450,0,3,False,False,NaN,None,...,gültig,06040808,06040808,Hüttenweg,1.3510,16.0,1,0.974803,0.829389,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16166,4557450,3277150,"POLYGON ((4557500 3277100, 4557500 3277200, 45...",CRS3035RES100mN3277150E4557450,0,16166,False,False,NaN,None,...,gültig,11020513,11020513,Orankesee,1.1997,36.0,1,0.944306,0.766277,NaN
16166,4557450,3277150,"POLYGON ((4557500 3277100, 4557500 3277200, 45...",CRS3035RES100mN3277150E4557450,0,16166,False,False,NaN,None,...,gültig,03051017,03051017,Rennbahnstraße,-0.0037,251.0,5,0.701734,0.766277,NaN
16167,4557550,3277150,"POLYGON ((4557600 3277100, 4557600 3277200, 45...",CRS3035RES100mN3277150E4557550,0,16167,False,False,NaN,None,...,gültig,03051017,03051017,Rennbahnstraße,-0.0037,251.0,5,0.701734,0.766277,NaN
16168,4557650,3277150,"POLYGON ((4557700 3277100, 4557700 3277200, 45...",CRS3035RES100mN3277150E4557650,0,16168,False,False,NaN,None,...,gültig,03051017,03051017,Rennbahnstraße,-0.0037,251.0,5,0.701734,0.766277,NaN
